In [ ]:
import torch
torch.__version__

In [ ]:
torch.manual_seed(123)

In [ ]:
f = torch.rand(2,3,3,4)

In [ ]:
f.dim()

In [ ]:
f

In [ ]:
# View

In [ ]:
f[0, 2, 1,3] == f[0][2][1][3] # returns a view

In [ ]:
t = f.view(3,4,6)

In [ ]:
t

In [ ]:
t.storage().data_ptr() == f.storage().data_ptr()

In [ ]:
t[1][2][1]

In [ ]:
f[0, 1:, :2, 0:3:2]

In [ ]:
# Reshape

In [ ]:
a = f.detach().clone()

In [ ]:
a

In [ ]:
n = a.reshape(6,3,4)

In [ ]:
n.storage().data_ptr() == a.storage().data_ptr()

In [ ]:
n.reshape((4,-1)).shape

In [ ]:
# Transpose

In [ ]:
ft = f.T

In [ ]:
f.shape

In [ ]:
ft.shape

In [ ]:
n.T.shape

In [ ]:
f.storage().data_ptr() == ft.storage().data_ptr(), n.storage().data_ptr() == n.T.storage().data_ptr()

In [ ]:
# batched transpose
bt = torch.zeros(n.shape[0], n.shape[2], n.shape[1])
for i in range(n.shape[0]):
    bt[i, :, :] = n[i, :, :].T

In [ ]:
bt2 = torch.transpose(n, 1, 2)

In [ ]:
n.shape

In [ ]:
bt2.shape

In [ ]:
bt.equal(bt2)

In [ ]:
bt.storage().data_ptr() == n.storage().data_ptr() # False

In [ ]:
bt2.storage().data_ptr() == n.storage().data_ptr() # False

In [ ]:
# Permute

In [ ]:
f.shape

In [ ]:
f.permute([3,2,0,1]).shape

In [ ]:
# Batched Matrix Multiplication

In [ ]:
x = torch.randn(3,5)
y = torch.randn(5,4)

In [ ]:
# vector x vector
t1 = torch.randn(3)
t2 = torch.randn(3)
torch.matmul(t1, t2).shape #torch.Size([])

In [ ]:
# matrix x vector
t1 = torch.randn(3,4)
t2 = torch.randn(4)
torch.matmul(t1, t2).shape #torch.Size([3])

In [ ]:
# batched matrix x broadcasted vector
t1 = torch.randn(10, 3, 4)
t2 = torch.randn(4)
torch.matmul(t1, t2).shape #torch.size([10,3])

In [ ]:
# batched matirx x batched matrix
t1 = torch.randn(10, 3, 4)
t2 = torch.randn(10, 4, 5)
torch.matmul(t1, t2).shape #torch.size([10,3,5])

In [ ]:
# batched matrix x broadcasted matrix
t1 = torch.randn(10, 3, 4)
t2 = torch.rand(4, 5)
torch.matmul(t1, t2).shape #torch.size([10,3,5])

In [ ]:
# Neural Network

In [ ]:
import torch.nn.functional as F
from torch.autograd import grad

torch.manual_seed(12)
y = torch.randint(low=0, high=2, size=(10,))
x1 = torch.randn(10)
w1 = torch.randn(10, requires_grad=True)
b = torch.randn(10, requires_grad=True)

z = w1 * x1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y.type(torch.float32))
print(loss)

# grad_L_w1 = grad(loss, w1, retain_graph=True)
# grad_L_b = grad(loss, b, retain_graph=True)
# print(grad_L_w1)
# print(grad_L_b)

loss.backward()

print(w1.grad)
print(b.grad)

In [ ]:
class Model(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
          # 1st hidden layer
          torch.nn.Linear(num_inputs, 30),
          # Activation layer
          torch.nn.ReLU(),
          # 2nd hidden layer
          torch.nn.Linear(30, 20),
          torch.nn.ReLU(),
          # output
          torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [ ]:
torch.manual_seed(12)
model = Model(50, 3)

In [ ]:
print(model)

In [ ]:
print([p for p in model.parameters()])

In [ ]:
print(model.layers[0].weight.shape)
print(model.layers[2].weight.shape)
print(model.layers[4].weight.shape)

In [ ]:
# Model training
torch.manual_seed(12)
model = model
optimizer = torch.optim.SGD(model.parameters(), lr=0.4)

num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")